# 🐍 Python para Ingeniería de Datos — BSG Institute
## Sesión 9 · Bloque 3.5.x
### Exposición de datos con FastAPI

---

**Lo que veremos hoy:**
1. ¿Qué es una API y por qué la necesitamos?
2. Estructura de `api.py` — cómo está organizado
3. Probar cada endpoint desde Python con `httpx`
4. Entender path params vs query params
5. Leer la documentación automática (Swagger UI)

---

> ⚠️ **Antes de ejecutar este notebook:**
> 1. Activa el venv: `source venv/bin/activate`
> 2. Levanta la API en otra terminal: `uvicorn api:app --reload --port 8000`
> 3. Verifica que ves `Application startup complete` en esa terminal
> 4. Abre http://localhost:8000/docs en el browser
> 5. Luego regresa aquí y ejecuta las celdas

---
## 🧠 CONCEPTO: ¿Qué es una API y por qué la necesitamos?

En las sesiones anteriores construimos un pipeline que:
- Lee datos crudos
- Los limpia
- Los guarda en MySQL

Pero esos datos están **atrapados** en la base de datos. Solo quien tiene las credenciales puede acceder.

Una **API REST** es la puerta de salida: permite que cualquier aplicación — un dashboard, una app móvil, otro sistema — consuma los datos **sin necesitar acceso directo a MySQL**.

> 💡 **Analogía:** El pipeline es la cocina del restaurante. La API es el mesero — recibe el pedido, va a la cocina, y trae el resultado. El cliente nunca entra a la cocina.

**El contrato de una API:**
- Tú defines **endpoints** (URLs) con un formato de respuesta garantizado
- Quien consume la API no necesita saber cómo está implementada
- Si cambias MySQL por otro motor mañana, la API sigue funcionando igual

---
## 🔌 CELDA 1 — Verificar que la API está viva

In [ ]:
import httpx
import json

# URL base de nuestra API corriendo en local
BASE_URL = 'http://localhost:8000'

def pretty(response):
    """Imprime la respuesta de forma legible."""
    print(f'Status: {response.status_code}')
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))

# Health check — el primer endpoint que siempre debes probar
r = httpx.get(f'{BASE_URL}/health')
pretty(r)

---
## 📊 CELDA 2 — Endpoint: Resumen general

El endpoint `/resumen` no recibe parámetros — siempre retorna las métricas globales.
Es un **GET sin parámetros**: la URL es suficiente para saber qué queremos.

In [ ]:
r = httpx.get(f'{BASE_URL}/resumen')
pretty(r)

# Acceder a valores individuales de la respuesta
data = r.json()
print(f'\n💰 Ventas totales  : ${data["ventas_totales"]:,.2f}')
print(f'🎫 Ticket promedio : ${data["ticket_promedio"]:,.2f}')
print(f'📦 Transacciones   : {data["total_transacciones"]}')

---
## 🔍 CELDA 3 — Endpoint: Listar transacciones (sin filtros)

Sin parámetros retorna las últimas 20 transacciones por defecto.

In [ ]:
r = httpx.get(f'{BASE_URL}/transacciones')
data = r.json()

print(f'Status: {r.status_code}')
print(f'Total retornado: {data["total"]}')
print(f'\nPrimeras 3 transacciones:')
for t in data['transacciones'][:3]:
    print(f'  {t}')

---
## 🎛️ CELDA 4 — Query parameters: filtrar por status

Los **query parameters** van después del `?` en la URL.
Son opcionales y permiten filtrar, paginar o modificar la respuesta.

Con `httpx` los pasas como diccionario en `params=` — no necesitas construir la URL a mano.

In [ ]:
# Filtrar solo transacciones COMPLETADAS
r = httpx.get(f'{BASE_URL}/transacciones', params={'status': 'COMPLETADA', 'limite': 5})
data = r.json()

# La URL que httpx construyó internamente:
print(f'URL consultada: {r.url}')
print(f'Filtros aplicados: {data["filtros"]}')
print(f'Resultados: {data["total"]}')
print()
for t in data['transacciones']:
    print(f'  {t["id_transaccion"]} | {t["fecha"]} | ${t["amount"]:>8.2f} | {t["store"]}')

In [ ]:
# Combinar dos filtros: status + sucursal
r = httpx.get(f'{BASE_URL}/transacciones', params={
    'status':   'COMPLETADA',
    'sucursal': 'CDMX-Norte',
    'limite':   10
})
data = r.json()
print(f'URL: {r.url}')
print(f'Resultados con dos filtros: {data["total"]}')
for t in data['transacciones']:
    print(f'  {t["id_transaccion"]} | ${t["amount"]:>8.2f}')

---
## 🎯 CELDA 5 — Path parameters: buscar por ID

Los **path parameters** van dentro de la URL, no después del `?`.
Se usan cuando buscas un recurso específico e identificable.

```
Query param  → /transacciones?status=COMPLETADA   (filtro, puede retornar varios)
Path param   → /transacciones/TXN-00001           (un ID exacto, retorna uno)
```

In [ ]:
# Buscar una transacción específica por su ID
id_buscar = 'TXN-00001'
r = httpx.get(f'{BASE_URL}/transacciones/{id_buscar}')

print(f'Status: {r.status_code}')
pretty(r)

In [ ]:
# ¿Qué pasa si el ID no existe? → debe retornar 404
r = httpx.get(f'{BASE_URL}/transacciones/TXN-99999')
print(f'Status: {r.status_code}')  # esperamos 404
print(r.json())

---
## 📈 CELDA 6 — Métricas por sucursal y por mes

In [ ]:
# Métricas por sucursal
r = httpx.get(f'{BASE_URL}/metricas/sucursales')
data = r.json()

print('📊 KPIs por sucursal (solo COMPLETADAS):')
print(f'{"Sucursal":<18} {"Transac.":>10} {"Ventas":>12} {"Ticket":>12}')
print('-' * 56)
for s in data['sucursales']:
    print(f'{s["sucursal"]:<18} {s["total_transacciones"]:>10} ${s["ventas_totales"]:>11,.2f} ${s["ticket_promedio"]:>11,.2f}')

In [ ]:
# Métricas por mes
r = httpx.get(f'{BASE_URL}/metricas/mensual')
data = r.json()

print('📅 KPIs por mes:')
print(f'{"Año":<6} {"Mes":>5} {"Transac.":>10} {"Ventas":>12}')
print('-' * 38)
for m in data['meses']:
    print(f'{m["anio"]:<6} {m["mes"]:>5} {m["total_transacciones"]:>10} ${m["ventas_totales"]:>11,.2f}')

---
## 🗺️ CELDA 7 — Mapa de todos los endpoints

También puedes consultar qué endpoints existen directamente desde Python.

In [ ]:
# El schema OpenAPI de FastAPI se expone en /openapi.json
r = httpx.get(f'{BASE_URL}/openapi.json')
schema = r.json()

print(f'API: {schema["info"]["title"]} v{schema["info"]["version"]}')
print(f'\nEndpoints disponibles:')
for path, methods in schema['paths'].items():
    for method, info in methods.items():
        print(f'  {method.upper():<6} {path:<35} → {info["summary"]}')

---
## 🌐 CELDA 8 — Swagger UI: documentación interactiva

FastAPI genera automáticamente una interfaz web para probar la API.
Ábrela en el browser mientras tienes la API corriendo:

**http://localhost:8000/docs**

Desde ahí puedes:
- Ver todos los endpoints con su descripción
- Probarlos con un formulario visual (sin escribir código)
- Ver los ejemplos de respuesta
- Compartirla con quien consume la API

> 💡 Esta documentación es **automática** — FastAPI la genera a partir de tu código. Si agregas un endpoint nuevo, aparece solo.

In [ ]:
# Abrir Swagger UI desde el notebook
import webbrowser
webbrowser.open('http://localhost:8000/docs')
print('✅ Swagger UI abierto en el browser')

---
## ✅ Resumen: ¿qué construiste hoy?

| Concepto | Dónde lo viste | Ejemplo |
|---|---|---|
| Endpoint GET sin params | `/health`, `/resumen` | Siempre retorna lo mismo |
| Query parameters | `/transacciones?status=COMPLETADA` | Filtros opcionales |
| Path parameters | `/transacciones/TXN-00001` | Recurso específico |
| Códigos HTTP | `200 OK`, `404 Not Found` | El contrato de la API |
| Swagger UI | `localhost:8000/docs` | Documentación automática |
| Conexión API → MySQL | Todas las rutas | El DE conecta las capas |

---
## 📝 TAREA — Para la próxima sesión

Añade **dos endpoints nuevos** a `api.py`:

**Endpoint A — `/clientes/{customer_id}`**
- Recibe un `customer_id` como path parameter
- Retorna todas las transacciones de ese cliente
- Si el cliente no tiene transacciones, retorna `404`

**Endpoint B — `/metricas/top-sucursal`**
- Sin parámetros
- Retorna solo la sucursal con mayor venta total (una sola fila)
- Pista: usa `ORDER BY ventas_totales DESC LIMIT 1`

**Entrega:** El archivo `api.py` modificado con los dos endpoints funcionando.
Pruébalos desde Swagger UI y toma un screenshot de cada uno respondiendo correctamente.

> 💡 El patrón es exactamente el mismo que los endpoints que ya existen. Copia, adapta, prueba.